# 02 · Teacher features, prefix isolation and lineage


A frozen teacher supplies features without learning from this predictor study.
Its input encoding must exclude future information before attention mixes tokens.
Its target encoding deliberately uses the full clip. These are different roles.

**Run this notebook independently in a fresh kernel.** The default
`teach` mode uses small generated examples. Set `FI_TUTORIAL_MODE=inspect`
and `FI_RUN_ROOT` before starting the kernel to read saved artifacts.
Set `FI_TUTORIAL_MODE=execute` with an explicit `FI_RUN_ROOT` to run the
production stages below. Execute notebooks **00 → 04** in order for the
full Experiment 0; each uses a fresh kernel and the same run directory.
Use the [notebook HAIC launchers](../../../slurm/future-innovation/NOTEBOOKS.md)
for scheduled execution. Inspection remains read-only. An absent local
file says nothing about the current state of a remote HAIC job.

[Study overview](../../../docs/studies/future-innovation/README.md) ·
[Historical direct-v2 specification](../../../docs/studies/future-innovation/direct-gate-protocol.md) ·
[Calibrated direct-v3 specification](../../../docs/studies/future-innovation/direct-v3-repair-protocol.md)

In [ ]:
from pathlib import Path
import os
import sys
from time import perf_counter

started = perf_counter()
override = os.environ.get("GAVD6_ROOT")
if override:
    candidates = [Path(override).expanduser().resolve()]
else:
    candidates = []
    for base in (Path.cwd(), *Path.cwd().parents):
        candidates.extend((base, base / "gavd6", base / "experiments/sjepa/gavd6"))
PROJECT_ROOT = next((p for p in candidates if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the checkout containing src/gavd6_sjepa.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from matplotlib_inline.backend_inline import set_matplotlib_formats
get_ipython().run_line_magic("matplotlib", "inline")
set_matplotlib_formats("svg", "png")
plt.rcParams.update({"figure.figsize": (8, 3), "axes.spines.top": False,
                    "axes.spines.right": False, "font.size": 11})

from gavd6_sjepa.research_directions.future_innovation.fi_tutorial_inspection import (
    artifact_inventory, inspect_report, read_optional_table, inspection_audit_path,
)

# Use "execute" for real stages or "inspect" for saved artifacts.
# Relative paths resolve from GAVD6_ROOT. Execution requires an explicit run root.
MODE = os.environ.get("FI_TUTORIAL_MODE", "teach")
if MODE not in {"teach", "inspect", "execute"}:
    raise ValueError("FI_TUTORIAL_MODE must be teach, inspect, or execute.")
if MODE == "execute" and not os.environ.get("FI_RUN_ROOT"):
    raise ValueError("Set FI_RUN_ROOT explicitly before executing real experiment stages.")
RUN_ROOT = Path(os.environ.get("FI_RUN_ROOT", "outputs/future-innovation-direct-v3-dev-20260911")).expanduser()
if not RUN_ROOT.is_absolute():
    RUN_ROOT = PROJECT_ROOT / RUN_ROOT
RUN_ROOT = RUN_ROOT.resolve()
print("Teaching examples only; no empirical gait findings." if MODE == "teach"
      else f"{MODE.upper()} mode: {RUN_ROOT}")
if MODE == "execute":
    from gavd6_sjepa.research_directions.future_innovation.fi_notebook_workflow import (
        initialize_from_environment, run_stage, build_notebook_report, finish_notebook_report,
        attempt_stage, require_stage_success,
    )
    if (RUN_ROOT / "config/run-contract.json").is_file():
        import json
        saved_run = json.loads((RUN_ROOT / "config/run-contract.json").read_text())
        print("Frozen protocol:", saved_run.get("protocol", "legacy-v1"),
              "— gate clips:", saved_run.get("cohort_size"))
        if saved_run.get("protocol", "legacy-v1") == "legacy-v1":
            print("This run retains legacy selectivity gates. Use a new run root for direct-v2.")

## Execute this stage

Cache the frozen teacher features and check input integrity, teacher repeatability and future isolation. The new direct-v2 protocol performs no person/background replacement or selectivity tests. Original encoding normally uses one H100. Cached direct-v3 verifies inherited arrays and audits entirely on CPU. A verified audit rejection finishes with TRAINING BLOCKED and no fitting; missing or corrupt evidence remains an execution error. Completed stages are verified and reused without loading the teacher again.

This cell runs only in `execute` mode. Each command uses this kernel's Python and the existing production CLI; stage logs are retained alongside the executed notebook.

In [ ]:
if MODE == "execute":
    device = os.environ.get("FI_NOTEBOOK_DEVICE", "cuda")
    stage_error = attempt_stage("cache-teacher", RUN_ROOT, "--device", device)
    if stage_error is None:
        stage_error = attempt_stage("audit-teacher", RUN_ROOT, "--device", device)

At 384 × 384 pixels, 16-pixel patches form a 24 × 24 grid. Two-frame tubelets
cover the temporal axis. Prefix encoding keeps only frames 0–31 before attention;
masking after full-clip attention would already contain future information.
The target pools the person region at frames 38–39 and applies the same frozen
256-dimensional projection in every fold.

RGB inputs concatenate global prefix pooling, the last prefix person region,
prefix background pooling, and nuisance summaries. Unavailable background
measurements have separate support fractions. Recording conditions may help
explain the contextual target, so a gain is not automatically a gait-dynamics claim.

In [ ]:
from gavd6_sjepa.research_directions.future_innovation.fi_contracts import FRAME
from gavd6_sjepa.research_directions.future_innovation.fi_token_regions import context_indices
past_ids=context_indices()
assert past_ids[-1] < FRAME.context_stop_exclusive // FRAME.tubelet_size * FRAME.grid**2
display(pd.DataFrame({'tokens':[len(past_ids),FRAME.frames_per_clip//FRAME.tubelet_size*FRAME.grid**2]},
                     index=['Prefix before attention','Full target context']))

Direct-v2 retained three preselected windows for repeatability and randomized
future-pixel checks. Direct-v3 reuses those measurements and cached arrays after
checking their original contracts, file hashes, identities, shapes and audit
arithmetic. It needs neither raw-video mounts nor a GPU. It does not rerun
teacher encoding or certify person/background selectivity. Legacy-v1 still
requires its original selectivity measurements.

| Evidence | What it establishes |
|---|---|
| Cache and projection hashes | Identity of reused arrays and projection |
| Repeated inference measurements | Teacher numerical stability on audited windows |
| Randomized future-pixel measurements | Prefix feature isolation on audited windows |
| Training target variance | Usable dimensions, with a training-only mask |
| Original cohort receipts | Inherited alignment and pose provenance |

In [ ]:
if MODE != "teach":
    from gavd6_sjepa.research_directions.future_innovation.fi_contracts import read_json
    lineage=RUN_ROOT/'config/parent-lineage.json'
    if lineage.is_file():
        record=read_json(lineage)
        display({k:record[k] for k in ['parent_root','parent_run_id','parent_protocol','teacher_evidence','raw_alignment_evidence','identities']})
    audit=inspection_audit_path(RUN_ROOT)
    if audit.is_file():
        record=read_json(audit)
        display(pd.DataFrame(list(record.get('checks',{}).items()),columns=['saved check','passed']))
        print('Teacher evidence:',record.get('teacher_evidence','original run audit; consult saved protocol'))
        print('Failed checks:',[k for k,v in record.get('checks',{}).items() if not v])
    for name in ['teacher-stability.csv','causal-leakage.csv','target-sensitivity.csv']:
        table=read_optional_table(RUN_ROOT,'qc/'+name)
        if table is not None:
            print(name); display(table)
    print('Inspect mode reads saved audit evidence; execution verifies the applicable stage.')

In [ ]:
if MODE == "execute":
    require_stage_success(stage_error)

## What this step establishes

The cached predictor inputs and contextual targets have explicit lineage. Next fit matched models using only training sources and expose each candidate’s validation evidence.

Continue with [03_matched_predictors_and_controls.ipynb](03_matched_predictors_and_controls.ipynb).

In [ ]:
print(f"Notebook elapsed time: {perf_counter() - started:.2f} seconds ({MODE} mode).")